(ground-loops)=
# Ground loops

## Voltage needs a reference

When you measure a voltage, you compare two points. A statement such as "this node is at $2\,\mathrm{V}$" leaves the second point unstated. Usually you mean that the node is $2\,\mathrm{V}$ above the circuit ground.

If $U(A)$ and $U(B)$ describe the potentials of two nodes relative to the same reference, the voltage from $A$ to $B$ is

$$
U_{AB}=U(A)-U(B).
$$

Calling a point **ground** chooses a reference. It does not make current disappear there. Current still needs a complete path back to its source. A ground wire also obeys Ohm's law, just like any other conductor.

The [resistance recap](resistance.ipynb) explains how to apply Ohm's law and Kirchhoff's laws. Here you apply the same reasoning to the wires and instrument connections that a simple circuit drawing often leaves out.

## What does ground mean?

You encounter several related uses of the word:

| Term | Meaning |
| --- | --- |
| Signal ground or circuit common | The reference node for the signal and often a return path for signal or supply current. |
| Chassis ground | The conductive enclosure or frame of an instrument. It can connect to circuit common, protective earth, or both, depending on the design. |
| Protective earth, PE | The protective conductor that connects exposed conductive parts of suitable mains-powered equipment to the building earthing system. It provides a path for fault current. |

These names describe functions. They do not tell you which points are connected inside a particular instrument. At our benches, the generator return and the oscilloscope BNC shields connect to protective earth. The scope channels also share a common reference inside the instrument.

A battery-powered circuit can have its own ground without a conducting connection to earth. Its voltage relative to earth is then not fixed by that connection. This is called **floating** operation. A USB cable or a grounded probe can add a reference path that was absent before. Check the complete setup before treating a source or a measurement input as floating. [Tektronix describes these grounding and isolation distinctions](https://www.tek.com/en/documents/application-note/three-facets-floating-measurement-solutions).

(ground-loop-formation)=
## How a ground loop forms

Suppose you connect a grounded signal source to a grounded measuring instrument. The signal cable includes a return conductor, often its shield. That conductor joins the source reference $G_s$ to the measurement reference $G_m$.

The two references already connect through the instruments' chassis and protective-earth wiring. Adding the cable return therefore creates a second conducting route between the same points. You can follow one route out and the other route back. This closed path is a **ground loop**.

```{figure} ../images/ground-loop-paths.svg
---
name: fig-appendix-ground-loop-paths
height: 360px
---
The cable return and the protective-earth wiring provide two paths between the source and instrument references.
```

The extra path can include equipment that seems unrelated to the measurement, such as a computer connected through USB. Draw these connections before deciding where the return current flows.

A loop does not automatically produce a visible error. There must also be a voltage that drives current around it, or a changing magnetic field that induces a voltage in the loop. The size of the resulting measurement error depends on the wiring and the signal you measure. [NI illustrates the two-reference measurement problem](https://www.ni.com/en/shop/data-acquisition/measurement-fundamentals/field-wiring-and-noise-considerations-for-analog-signals.html).

## Ground wires have resistance

In an ideal circuit diagram, a wire has zero resistance. Two points joined by that wire have exactly the same potential. Real wires and contacts have a small resistance, so a current through them produces a voltage drop.

To see how this matters, use the simplified model in {numref}`fig-appendix-ground-loop-model`. $R_g$ represents the cable return. $R_{other}$ represents the rest of the loop. The equivalent voltage $U_{loop}$ represents the disturbance driving the loop current. It is a modelling element, not an instruction to connect another source to the bench earth wiring.

```{figure} ../images/ground-loop-model.svg
---
name: fig-appendix-ground-loop-model
height: 340px
---
A driving voltage produces loop current through both return-path resistances, creating a voltage difference between $G_s$ and $G_m$.
```

For this example, neglect the measuring instrument's input current and consider only the unwanted circulating current. Treat the wires as resistors. This is a low-frequency approximation; inductance and capacitance also matter for fast-changing signals.

### Find the circulating current

Follow the arrow around the loop. Kirchhoff's voltage law gives a voltage rise at the equivalent source and a drop across each resistance:

$$
U_{loop}-I_{loop}R_g-I_{loop}R_{other}=0.
$$

Collect the two resistance terms:

$$
U_{loop}=I_{loop}(R_g+R_{other}).
$$

Therefore,

$$
I_{loop}=\frac{U_{loop}}{R_g+R_{other}}.
$$

The arrow through $R_g$ points from $G_s$ to $G_m$. Ohm's law gives the difference between these two reference potentials:

$$
U_g=U(G_s)-U(G_m)=I_{loop}R_g.
$$

Both points are called ground, but they are separated by a real conductor carrying current. That is why they need not have exactly the same potential.

(ground-loop-measurement-error)=
### Find the measurement error

Let the signal node be $S$. The voltage you want is measured relative to the source's own reference:

$$
U_{signal}=U(S)-U(G_s).
$$

A ground-referenced instrument instead measures relative to $G_m$:

$$
U_{meas}=U(S)-U(G_m).
$$

Add and subtract $U(G_s)$ in the second expression:

$$
U_{meas}=[U(S)-U(G_s)]+[U(G_s)-U(G_m)].
$$

Substitute the definitions above:

$$
U_{meas}=U_{signal}+U_g.
$$

The voltage drop along the return conductor appears in the result. If the loop current changes direction, the sign of this error changes too. If it varies with time, it adds a time-dependent disturbance to the measured waveform.

### Worked example: a small signal and a small ground voltage

Take illustrative values

$$
U_{loop}=10\,\mathrm{mV},\qquad R_g=0.2\,\Omega,\qquad R_{other}=0.8\,\Omega.
$$

The loop current is

$$
I_{loop}=\frac{10\,\mathrm{mV}}{0.2\,\Omega+0.8\,\Omega}=10\,\mathrm{mA}.
$$

The voltage drop in the cable return is then

$$
U_g=(10\,\mathrm{mA})(0.2\,\Omega)=2\,\mathrm{mV}.
$$

For a sensor signal of $20\,\mathrm{mV}$, the instrument reports

$$
U_{meas}=20\,\mathrm{mV}+2\,\mathrm{mV}=22\,\mathrm{mV}.
$$

The relative error is

$$
\frac{U_{meas}-U_{signal}}{U_{signal}}=\frac{2\,\mathrm{mV}}{20\,\mathrm{mV}}=0.10.
$$

That is a 10 percent error, even though the unwanted voltage is only $2\,\mathrm{mV}$. The same unwanted voltage would matter much less when measuring a signal of several volts. These numbers describe the example circuit; they are not measurements of the bench wiring.

## Why a loop can pick up interference

The wiring in a ground loop encloses an area. A changing magnetic flux through that area can induce a voltage around the loop. Nearby transformers, power cables, and switching circuits can supply changing magnetic fields. Reducing the enclosed area reduces this coupling for a given field orientation.

The unwanted voltage may contain a mains-frequency component, such as $50\,\mathrm{Hz}$ at our benches, or harmonics and switching-frequency components. A steady disturbance can instead produce an offset. A $50\,\mathrm{Hz}$ component alone does not prove that you have a ground loop: electric-field coupling and other interference mechanisms can also produce it. Use the wiring and a controlled comparison to identify the cause. [Tektronix discusses ground-loop noise and induced pickup](https://www.tek.com/en/documents/whitepaper/abcs-probes-primer).

### A shared return can also create an error

You do not need a second ground path for every grounding problem. Suppose a sensor and an LED share a section of return wire. Switching the LED changes the current through that section. Its voltage drop changes, so the sensor reference moves relative to the measuring instrument.

This is **common-impedance coupling**: two circuits share a conductor with nonzero impedance. A ground loop can produce this kind of error when its current flows through the signal return, but shared-return errors can occur without a ground loop. Tracing which currents use each wire helps you tell the cases apart.

(ground-loops-bridge)=
## Example: a bridge rectifier and two grounded instruments

The full bridge in [manual 5.1](../../manuals/week5/5.1_diode_circuits.ipynb) has two source terminals and two load terminals. The load current keeps the same direction during both source half-cycles. That does not make its negative terminal identical to the generator return.

In the correct circuit, $D_1$ and $D_2$ conduct during the positive source half-cycle. During the negative half-cycle, $D_4$ and $D_3$ conduct. The load sits between the two conducting diodes in either case.

Now consider a one-channel scope measurement with the probe tip on `out_plus` and its ground clip on `out_minus`.

```{figure} ../../manuals/week5/images/bridge-shared-ground.svg
---
name: fig-appendix-bridge-ground-fault
height: 400px
---
The misplaced scope ground clip and the shared protective-earth connection bypass $D_2$.
```

The ground clip connects to the scope chassis. The scope chassis connects through protective earth to the generator return. You have therefore added a conducting connection from `out_minus` to the generator return, across $D_2$.

This changes the circuit itself. It is more direct than adding a small interference voltage to a measurement. For the same diode orientations:

| Source half-cycle | Consequence of grounding `out_minus` |
| --- | --- |
| Positive | Current reaches the load through $D_1$ and returns through the added connection. The intended path loses the forward drop of $D_2$. |
| Negative | $D_3$ provides a path from the grounded load-negative node to the source terminal. This bypasses the load and can draw a large current. The output loses one of its two rectified pulses. |

For example, a source model with a $10\,\mathrm{V}$ negative peak, $50\,\Omega$ output resistance, and a $0.7\,\mathrm{V}$ diode drop gives an approximate fault current of

$$
I_{fault}\approx\frac{10\,\mathrm{V}-0.7\,\mathrm{V}}{50\,\Omega}=186\,\mathrm{mA}.
$$

The $1\,\mathrm{k}\Omega$ load does not limit that current because it is outside the fault path. A real generator may limit its output or enter protection. You analyse this incorrect connection on paper or in simulation.

The ALPACA LED bridge uses the same four-node arrangement. There, the misplaced connection bypasses the entire $D_2$ branch, including its internal current-limiting resistor. The extra resistors change the currents, but they do not make a misplaced ground clip a valid measurement connection.

(ground-loops-differential)=
## Measure the voltage between the load terminals

Use two scope channels to measure the load terminals relative to the same generator return. Put the CH1 tip on `out_plus` and the CH2 tip on `out_minus`. Connect both ground clips to the generator return.

```{figure} ../images/ground-loop-bridge-differential.svg
---
name: fig-appendix-bridge-differential
height: 400px
---
Both probe tips measure relative to the generator return; CH1 minus CH2 gives the voltage across the load. Crossings without a dot are not connections.
```

Each channel measures a voltage relative to $G$, the generator return:

$$
U_{CH1}=U(out\_plus)-U(G),
$$

$$
U_{CH2}=U(out\_minus)-U(G).
$$

Subtract the readings at the same instant:

$$
U_{CH1}-U_{CH2}=U(out\_plus)-U(out\_minus)=U_{out}.
$$

The common reference cancels algebraically. For example, readings of $-2\,\mathrm{V}$ and $-5\,\mathrm{V}$ give a load voltage of $+3\,\mathrm{V}$. The load polarity can be positive even when both terminals are below the generator return.

Use matched probe attenuation, correct channel settings, and DC coupling for this comparison. Neither channel may clip. Subtracting two readings does not recover information already lost by an overloaded input. [Tektronix describes the two-passive-probe method](https://www.tek.com/en/support/faqs/how-can-i-make-differential-measurement-passive-probes).

### Differential measurement and isolation

A **differential measurement** responds to the voltage difference between two inputs. **Electrical isolation** removes a direct conducting connection across an isolation barrier. These are different properties.

Scope subtraction does not disconnect either ground clip or provide input isolation. It also does not guarantee rejection of all common-mode interference: probe mismatch and channel errors limit the cancellation. A differential probe can provide better rejection, but you still need to check its input-to-ground limits and isolation specification.

The two-channel method above is useful for the low-frequency, low-voltage bridge signals in this practicum. It is not a general method for measuring arbitrary voltages above earth. Use measurement equipment rated for the individual input voltages as well as their difference. [The Tektronix probe primer explains the limits of channel subtraction](https://www.tek.com/en/documents/whitepaper/abcs-probes-primer).

```{attention}
Keep protective earth connected. Disconnecting it can put the scope enclosure and connectors at the potential of a probe ground clip. Use the appropriate measurement connections or rated isolated equipment; do not alter the instrument's protective-earth wiring.
```

## Reduce grounding errors in practice

Choose the remedy from the current path that causes the problem.

* **Use one deliberate circuit reference point for the bench connections.** At low frequencies, returning sensitive signal wiring and larger supply currents separately to that point reduces shared voltage drops. Connecting everything somewhere on a long breadboard ground rail does not make the rail ideal.
* **Keep a signal and its intended return close together.** Short paired wiring reduces the loop area available for magnetic pickup. Keep it away from mains leads and transformer wiring.
* **Measure across the points that define the signal.** A suitable differential input can reject a voltage that appears equally on both inputs. Its common-mode range and rejection are finite, so check both.
* **Use isolation where the complete setup needs it.** A rated isolated source, measurement input, or interface can break a conducting path without removing protective earth. Check whether another connection, such as USB, restores that path.

A cable shield may also be the signal return. Disconnecting it can force current to take another route through the equipment, so removing shields is not a universal ground-loop remedy. Likewise, putting both scope ground clips at the correct reference prevents the bridge short, but it does not remove every possible loop elsewhere in the setup.

## Check a suspected ground loop

Start by drawing the complete measurement circuit. Include probe ground clips, BNC shields, supply returns, USB connections, and any known chassis-to-earth connections. Look for two routes joining the same reference points and for ground clips joining nodes that should stay distinct.

You can check accessible ground connections with the DMM in continuity or resistance mode. Disconnect signal leads and probes from the circuit first. Switch off the instruments and leave their power cords connected when checking the protective-earth path between their accessible BNC shields. Do not use resistance mode on an energised circuit. The [grounding task in manual 5.1](Task_I5_5_1) applies this check to the generator and scope.

A continuity beep establishes that a conducting path exists. It does not prove that the path has zero impedance or that it carries no unwanted current during a measurement.

After restoring the correct connections, change one relevant condition at a time. For example, keep the circuit and scope settings fixed while reducing the area enclosed by the signal and return wiring. Record whether the interference changes. Changing several cables and settings at once may improve the trace, but it leaves you without an explanation you can use next time.